# Write a protein back out, and read it with openff-pablo unchanged

Hen lysozyme from Pablo's corpus, which has four disulfides. `save_pdb`
writes a `CONECT` for each disulfide and for nothing else, and Pablo reads
the file with no extra arguments.

In [1]:
import logging, warnings
from rdkit import RDLogger

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
RDLogger.DisableLog("rdApp.*")

In [2]:
import urllib.request
from pathlib import Path

cache = Path("../assets_cache")
cache.mkdir(exist_ok=True)
source = cache / "193l_prepared.pdb"
if not source.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/openforcefield/openff-pablo/main/"
        "openff/pablo/_tests/data/prepared_pdbs/193l_prepared.pdb",
        source,
    )

In [3]:
from mbuild.biopolymers import Protein

protein = Protein(source)
print(len(list(protein.residues())), "residues,", protein.n_particles, "atoms,", protein.n_bonds, "bonds, net charge", protein.net_formal_charge)
for record in protein.bond_records():
    print(record["residue_names"], record["residue_numbers"], record["atom_names"], "leaving", record["leaving_atoms"])

129 residues, 1960 atoms, 1984 bonds, net charge 8
('CYS', 'CYS') (6, 127) ('SG', 'SG') leaving (['HG'], ['HG'])
('CYS', 'CYS') (30, 115) ('SG', 'SG') leaving (['HG'], ['HG'])
('CYS', 'CYS') (64, 80) ('SG', 'SG') leaving (['HG'], ['HG'])
('CYS', 'CYS') (76, 94) ('SG', 'SG') leaving (['HG'], ['HG'])


In [4]:
written = cache / "193l_mbuild.pdb"
protein.save_pdb(written, overwrite=True)

lines = written.read_text().splitlines()
print({kind: sum(line.startswith(kind) for line in lines) for kind in ("ATOM", "HETATM", "TER", "CONECT")})
print(*[line for line in lines if line.startswith("CONECT")], sep="\n")

{'ATOM': 1960, 'HETATM': 0, 'TER': 1, 'CONECT': 8}
CONECT  101 1916
CONECT  471 1734
CONECT  993 1220
CONECT 1163 1407
CONECT 1220  993
CONECT 1407 1163
CONECT 1734  471
CONECT 1916  101


In [5]:
from openff.pablo import topology_from_pdb

topology = topology_from_pdb(written)
print(topology.n_molecules, "molecule,", topology.n_atoms, "atoms,", topology.n_bonds, "bonds, net charge", topology.molecule(0).total_charge)
assert topology.n_bonds == protein.n_bonds

1 molecule, 1960 atoms, 1984 bonds, net charge 8.0 elementary_charge
